# Imports

In [206]:
from harbor.analysis import cross_docking as cd
import pandas as pd
from importlib import reload
reload(cd)

<module 'harbor.analysis.cross_docking' from '/Users/alexpayne/Scientific_Projects/harbor/harbor/analysis/cross_docking.py'>

In [89]:
raw_df = pd.read_csv("/Users/alexpayne/Scientific_Projects/mers-drug-discovery/sars2-retrospective-analysis/20250212_p_to_x_posit/20250311_combined_results") 

In [90]:
settings = cd.Settings()

In [91]:
settings.use_scaffold_split = True
settings.use_date_split = True
settings.use_random_split = False
settings.scaffold_split_option = cd.ScaffoldSplitOptions.X_TO_ALL
settings.query_scaffold_min_count = 5
settings.n_per_split = list(range(1, 11)) + list(range(15, len(df.Reference_Structure.unique())+20, 20))

AttributeError: 'DataFrame' object has no attribute 'Reference_Structure'

In [ ]:
settings.to_yml_file("scaffold_split_settings.yml")

In [ ]:
cluster_sizes = raw_df.groupby(settings.query_scaffold_id_column)[
    settings.query_ligand_column
].nunique()

query_subset_list = [
    [scaffold]
    for scaffold in cluster_sizes[
        cluster_sizes > settings.query_scaffold_min_count
    ].index.tolist()
]

In [ ]:
date_dict_list = (
                raw_df.groupby(settings.reference_structure_column)[
                    [
                        settings.reference_structure_column,
                        settings.reference_structure_date_column,
                    ]
                ]
                .head(1)
                .to_dict(orient="records")
            )

simplified_date_dict = {
    date_dict[settings.reference_structure_column]: date_dict[
        settings.reference_structure_date_column
    ]
    for date_dict in date_dict_list
}

In [103]:
settings.n_per_split = [1,5,20,100,315]
evs = [cd.Evaluator(dataset_split=cd.ScaffoldSplit(query_scaffold_id_column=settings.query_scaffold_id_column, 
                                                   reference_scaffold_id_column=settings.reference_scaffold_id_column, 
                                                   query_scaffold_id_subset=query_subset, 
                                                   split_option = settings.scaffold_split_option,
                                                   n_per_split=-1,),
                    extra_splits=[cd.DateSplit(reference_structure_column=settings.reference_structure_column, 
                                              n_per_split=n_per_split,
                                              balanced=True,
                                              date_dict=simplified_date_dict,
                                              randomize_by_n_days=settings.randomize_by_n_days,
                                              )],
                    scorer=scorer,
                    evaluator=cd.BinaryEvaluation(variable="RMSD", cutoff=2.0),
                    groupby=[settings.query_ligand_column]
                    
                    )   
       for query_subset in query_subset_list
       for n_per_split in settings.n_per_split
       for scorer in [cd.POSITScorer()]
       ]

In [104]:
import tqdm
results = []
for ev in tqdm.tqdm(evs):
    results.append(cd.Results(evaluator=ev, fraction_good=ev.run(raw_df)))

 16%|█▌        | 4/25 [00:22<01:57,  5.60s/it]


KeyboardInterrupt: 

In [106]:
df = cd.Results.df_from_results(results)

In [131]:
df

,Bootstraps,StructureChoice,StructureChoice_Choose_N,Score,Score_Choose_N,EvaluationMetric,EvaluationMetric_Cutoff,Split,N_Per_Split,Query_Scaffold_ID_Column,...,Split_1,N_Per_Split_1,Reference_Structure_Column_1,Min,Max,CI_Upper,CI_Lower,Total,Fraction,Scaffold
0,1,Dock_to_All,All,POSIT,1,RMSD,2.0,ScaffoldSplit,-1,cluster_id,...,DateSplit,1,Reference_Structure,0.000000,0.000000,0.053570,0.000378,66,0.000000,0
1,1,Dock_to_All,All,POSIT,1,RMSD,2.0,ScaffoldSplit,-1,cluster_id,...,DateSplit,5,Reference_Structure,0.772727,0.772727,0.856912,0.657767,66,0.772727,0
2,1,Dock_to_All,All,POSIT,1,RMSD,2.0,ScaffoldSplit,-1,cluster_id,...,DateSplit,20,Reference_Structure,0.787879,0.787879,0.868941,0.674337,66,0.787879,0
3,1,Dock_to_All,All,POSIT,1,RMSD,2.0,ScaffoldSplit,-1,cluster_id,...,DateSplit,100,Reference_Structure,0.651515,0.651515,0.755305,0.530617,66,0.651515,0


In [107]:
df['Scaffold'] = df.Query_Scaffold_ID_Subset.apply(lambda x: x[0])

In [108]:
import plotly.express as px

In [111]:
fig = px.line(df, x="N_Per_Split_1", 
              y="Fraction", 
              color="Scaffold", 
              facet_col="Score", 
              template='simple_white',
              log_x=True,
              height=400,
              width=400)

In [112]:
fig.show()

In [195]:
dfs = []
testdf = raw_df.copy()
testdf = testdf[testdf.Pose_ID == 0]
testdf.sort_values('Reference_Structure_Date', inplace=True)
structure_list = testdf.groupby('Reference_Structure').head(1).Reference_Structure.to_list()
structure_lists = [structure_list[:i] for i in range(1, len(structure_list)+1, 10)]
lengths = [len(structure_list) for structure_list in structure_lists]
subset_list = [x[0] for x in query_subset_list]
print('running over clusters')
for cluster_id in subset_list:
    print(cluster_id)
    clusterdf = testdf[testdf.cluster_id == cluster_id]
    by_n_per_split = []
    for structure_list in structure_lists:
        nps = clusterdf[clusterdf.Reference_Structure.isin(structure_list)]
        nps.sort_values("docking-confidence-POSIT", ascending=False, inplace=True)
        final = nps.groupby("Query_Ligand").head(1)
        by_n_per_split.append(sum(final.RMSD < 2.0) / len(final))
    dfs.append(pd.DataFrame({'N_Per_Split': lengths, 'Fraction': by_n_per_split, 'Scaffold': cluster_id}))

running over clusters
0


/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipykernel_9520/513728215.py:16: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipykernel_9520/513728215.py:16: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipykernel_9520/513728215.py:16: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipykernel

2


/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipykernel_9520/513728215.py:16: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipykernel_9520/513728215.py:16: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipykernel_9520/513728215.py:16: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipykernel

5


/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipykernel_9520/513728215.py:16: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipykernel_9520/513728215.py:16: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipykernel_9520/513728215.py:16: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipykernel

9


/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipykernel_9520/513728215.py:16: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipykernel_9520/513728215.py:16: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipykernel_9520/513728215.py:16: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipykernel

13


/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipykernel_9520/513728215.py:16: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipykernel_9520/513728215.py:16: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipykernel_9520/513728215.py:16: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipykernel

In [196]:
final_test_df = pd.concat(dfs)

In [199]:
fig = px.line(final_test_df, x="N_Per_Split", y="Fraction", color="Scaffold", log_x=True, template='simple_white', height=400, width=400)

/Users/alexpayne/miniforge-pypy3/envs/harbor/lib/python3.11/site-packages/plotly/express/_core.py:2065: FutureWarning:

When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.



In [200]:
fig.show()

# Don't dock to the same scaffold!

In [201]:
dfs = []
testdf = raw_df.copy()
testdf = testdf[testdf.Pose_ID == 0]
testdf.sort_values('Reference_Structure_Date', inplace=True)
structure_list = testdf.groupby('Reference_Structure').head(1).Reference_Structure.to_list()
structure_lists = [structure_list[:i] for i in range(1, len(structure_list)+1, 10)]
lengths = [len(structure_list) for structure_list in structure_lists]
subset_list = [x[0] for x in query_subset_list]
print('running over clusters')
for cluster_id in subset_list:
    print(cluster_id)
    # this line is the only difference
    clusterdf = testdf[(testdf.cluster_id == cluster_id)&(testdf.cluster_id_Reference != cluster_id)]
    by_n_per_split = []
    for structure_list in structure_lists:
        nps = clusterdf[clusterdf.Reference_Structure.isin(structure_list)]
        nps.sort_values("docking-confidence-POSIT", ascending=False, inplace=True)
        final = nps.groupby("Query_Ligand").head(1)
        by_n_per_split.append(sum(final.RMSD < 2.0) / len(final))
    dfs.append(pd.DataFrame({'N_Per_Split': lengths, 'Fraction': by_n_per_split, 'Scaffold': cluster_id}))

running over clusters
0


/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipykernel_9520/3227127880.py:17: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipykernel_9520/3227127880.py:17: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipykernel_9520/3227127880.py:17: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipyker

2


/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipykernel_9520/3227127880.py:17: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipykernel_9520/3227127880.py:17: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipykernel_9520/3227127880.py:17: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipyker

5


/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipykernel_9520/3227127880.py:17: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipykernel_9520/3227127880.py:17: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipykernel_9520/3227127880.py:17: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipyker

9


/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipykernel_9520/3227127880.py:17: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipykernel_9520/3227127880.py:17: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipykernel_9520/3227127880.py:17: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipyker

13


/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipykernel_9520/3227127880.py:17: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipykernel_9520/3227127880.py:17: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipykernel_9520/3227127880.py:17: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/cf/d42qwdmd5_g5s63r3g_vgx2c0000gn/T/ipyker

In [202]:
final_test_df = pd.concat(dfs)

In [205]:
fig = px.line(final_test_df, x="N_Per_Split", y="Fraction", color="Scaffold", log_x=True, template='simple_white', height=600, width=600)

/Users/alexpayne/miniforge-pypy3/envs/harbor/lib/python3.11/site-packages/plotly/express/_core.py:2065: FutureWarning:

When grouping with a length-1 list-like, you will need to pass a length-1 tuple to get_group in a future version of pandas. Pass `(name,)` instead of `name` to silence this warning.



In [225]:
reload(cd)
settings = cd.Settings()

In [226]:
settings.use_scaffold_split = True

In [227]:
settings.update_n_per_split(raw_df)

In [228]:
dsplits = settings.create_dataset_splits(raw_df)

In [229]:
len(dsplits)

70

In [231]:
len(dsplits)

70

# combine dataset splits

In [232]:
from collections import defaultdict
split_dict = defaultdict(list)
for ds in dsplits:
    split_dict[ds.name].append(ds)

In [233]:
split_dict

defaultdict(list,
            {'RandomSplit': [RandomSplit(type_='RandomSplit', name='RandomSplit', n_splits=1, n_per_split=1, deterministic=False, split_level=0, reference_structure_column='Reference_Ligand'),
              RandomSplit(type_='RandomSplit', name='RandomSplit', n_splits=1, n_per_split=2, deterministic=False, split_level=0, reference_structure_column='Reference_Ligand'),
              RandomSplit(type_='RandomSplit', name='RandomSplit', n_splits=1, n_per_split=3, deterministic=False, split_level=0, reference_structure_column='Reference_Ligand'),
              RandomSplit(type_='RandomSplit', name='RandomSplit', n_splits=1, n_per_split=4, deterministic=False, split_level=0, reference_structure_column='Reference_Ligand'),
              RandomSplit(type_='RandomSplit', name='RandomSplit', n_splits=1, n_per_split=5, deterministic=False, split_level=0, reference_structure_column='Reference_Ligand'),
              RandomSplit(type_='RandomSplit', name='RandomSplit', n_splits=1

In [237]:
from itertools import product
combined_splits = []
for split1 in ['RandomSplit', 'DateSplit']:
    for split2 in ['SimilaritySplit', 'ScaffoldSplit']:
        split1s = split_dict[split1]
        split2s = split_dict[split2]
        for s1, s2 in product(split1s, split2s):
            combined_splits.append((s1, s2))

In [238]:
len(combined_splits)

1225

In [239]:
combined_splits[0]

(RandomSplit(type_='RandomSplit', name='RandomSplit', n_splits=1, n_per_split=1, deterministic=False, split_level=0, reference_structure_column='Reference_Ligand'),
 ScaffoldSplit(type_='ScaffoldSplit', name='ScaffoldSplit', n_splits=1, n_per_split=1, deterministic=True, split_level=0, query_scaffold_id_column='cluster_id', reference_scaffold_id_column='cluster_id_Reference', query_scaffold_id_subset=None, reference_scaffold_id_subset=None, split_option=<ScaffoldSplitOptions.X_TO_X: 'x_to_x'>))

In [444]:
reload(cd)
settings = cd.Settings()

In [445]:
settings.use_scaffold_split = True
settings.scaffold_split_option = 'x_to_not_x'
settings.query_scaffold_min_count = 5
settings.combine_core_and_chemical_splits = True

In [446]:
settings.update_n_per_split(raw_df)

In [447]:
dsplits = settings.create_dataset_splits(raw_df)

In [448]:
dataset_splits, extra_splits = settings.combine_splits(dsplits)

In [449]:
len(dataset_splits)

180

In [450]:
len(extra_splits)

180

In [451]:
evs = settings.create_evaluators(raw_df)

In [452]:
len(evs)

360

In [453]:
s2 = cd.Settings()

In [454]:
settings.similarity_range

[0, 1]

In [455]:
s2.similarity_range

[0, 1]

In [456]:
s2.similarity_range = (0,20)

In [457]:
settings.similarity_range

[0, 1]

In [458]:
settings.to_yaml_file("scaffold_split_settings.yml")

In [459]:
raw_df.Reference_Structure.nunique()

309

In [460]:
evs[-1]

Evaluator(type_='Evaluator', name='Evaluator', pose_selector=PoseSelector(type_='PoseSelector', name='Default', category='PoseSelection', variable='Pose_ID', higher_is_better=False, number_to_return=1, groupby=['Query_Ligand', 'Reference_Ligand']), dataset_split=RandomSplit(type_='RandomSplit', name='RandomSplit', n_splits=1, n_per_split=325, deterministic=False, split_level=0, reference_structure_column='Reference_Ligand'), extra_splits=[ScaffoldSplit(type_='ScaffoldSplit', name='ScaffoldSplit', n_splits=1, n_per_split=-1, deterministic=True, split_level=1, query_scaffold_id_column='cluster_id', reference_scaffold_id_column='cluster_id_Reference', query_scaffold_id_subset=[13], reference_scaffold_id_subset=None, split_option=<ScaffoldSplitOptions.X_TO_NOT_X: 'x_to_not_x'>)], structure_choice=StructureChoice(type_='StructureChoice', name='Dock_to_All', category='StructureChoice', variable='Tanimoto', higher_is_better=True, number_to_return=None), scorer=Scorer(type_='Scorer', name='RMS

In [461]:
evs[-1].run(raw_df)

ScaffoldSplitOptions.X_TO_NOT_X
ScaffoldSplitOptions.X_TO_NOT_X
ScaffoldSplitOptions.X_TO_NOT_X
ScaffoldSplitOptions.X_TO_NOT_X
ScaffoldSplitOptions.X_TO_NOT_X
ScaffoldSplitOptions.X_TO_NOT_X
ScaffoldSplitOptions.X_TO_NOT_X
ScaffoldSplitOptions.X_TO_NOT_X
ScaffoldSplitOptions.X_TO_NOT_X
ScaffoldSplitOptions.X_TO_NOT_X
ScaffoldSplitOptions.X_TO_NOT_X
ScaffoldSplitOptions.X_TO_NOT_X
ScaffoldSplitOptions.X_TO_NOT_X
ScaffoldSplitOptions.X_TO_NOT_X
ScaffoldSplitOptions.X_TO_NOT_X
ScaffoldSplitOptions.X_TO_NOT_X
ScaffoldSplitOptions.X_TO_NOT_X
ScaffoldSplitOptions.X_TO_NOT_X
ScaffoldSplitOptions.X_TO_NOT_X
ScaffoldSplitOptions.X_TO_NOT_X
ScaffoldSplitOptions.X_TO_NOT_X
ScaffoldSplitOptions.X_TO_NOT_X
ScaffoldSplitOptions.X_TO_NOT_X
ScaffoldSplitOptions.X_TO_NOT_X
ScaffoldSplitOptions.X_TO_NOT_X
ScaffoldSplitOptions.X_TO_NOT_X
ScaffoldSplitOptions.X_TO_NOT_X
ScaffoldSplitOptions.X_TO_NOT_X
ScaffoldSplitOptions.X_TO_NOT_X
ScaffoldSplitOptions.X_TO_NOT_X
ScaffoldSplitOptions.X_TO_NOT_X
Scaffold

FractionGood(type_='FractionGood', name='FractionGood', total=6, fraction=1.0, replicates=[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0,

In [467]:
testdf = raw_df.copy()

In [469]:
ev = evs[-1]
a = ev.pose_selector.run(testdf)

In [470]:
b = ev.dataset_split.run(a)[0]

In [471]:
c = ev.extra_splits[0].run(b)[0]

ScaffoldSplitOptions.X_TO_NOT_X


NotImplementedError: Split option ScaffoldSplitOptions.X_TO_NOT_X not implemented

In [ ]:
d = ev.structure_choice.run()

In [462]:
settings.scaffold_split_option in [cd.ScaffoldSplitOptions.X_TO_ALL, cd.ScaffoldSplitOptions.X_TO_NOT_X]

True

In [463]:
scaffold_split = evs[-1].extra_splits[0]

In [464]:
reload(cd)
ssdict = scaffold_split.model_dump_json()

In [465]:
reload(cd)
ss = cd.ScaffoldSplit(**scaffold_split.dict())

In [466]:
ss.run(raw_df)

ScaffoldSplitOptions.X_TO_NOT_X


[         Unnamed: 0        Query_Ligand Reference_Structure  \
 1077384     1077384  BEN-BAS-c2bc0d80-6      Mpro-x11317_0A   
 1077385     1077385  BEN-BAS-c2bc0d80-6      Mpro-x11317_0A   
 1077386     1077386  BEN-BAS-c2bc0d80-6      Mpro-x11317_0A   
 1077387     1077387  BEN-BAS-c2bc0d80-6      Mpro-x11317_0A   
 1077388     1077388  BEN-BAS-c2bc0d80-6      Mpro-x11317_0A   
 ...             ...                 ...                 ...   
 4950027     4950027  VLA-UCB-50c39ae8-2       Mpro-x0991_0A   
 4950028     4950028  VLA-UCB-50c39ae8-2       Mpro-x0991_0A   
 4950029     4950029  VLA-UCB-50c39ae8-2       Mpro-x0991_0A   
 4950030     4950030  VLA-UCB-50c39ae8-2       Mpro-x0991_0A   
 4950031     4950031  VLA-UCB-50c39ae8-2       Mpro-x0991_0A   
 
                  Reference_Ligand_SMILES  \
 1077384  Cc1c(cncc1NC(=O)Cc2cccc(c2)Cl)N   
 1077385  Cc1c(cncc1NC(=O)Cc2cccc(c2)Cl)N   
 1077386  Cc1c(cncc1NC(=O)Cc2cccc(c2)Cl)N   
 1077387  Cc1c(cncc1NC(=O)Cc2cccc(c2)Cl)N   
 1077